# Week 10 Lab: Naive Bayes Classifier


NAME: T. DEVI PRIYA

ROLL NO. : AP24110011665

SECTION: CSE - C


**CSE303 Machine Learning | Unit III: Probability and Bayes Learning | Syllabus Exp 9 (naive Bayesian classifier and its accuracy)**

Pattern: **demo → practice → predict-then-run → deliberate error → reflection**.

| Part | Topic | Track |
|---|---|---|
| 1 | Probability and Bayes' theorem | core |
| 2 | Naive Bayes by hand (Play Tennis), and the zero-frequency error | core |
| 3 | Gaussian Naive Bayes in sklearn, log-probabilities, and scaling | core |
| 4 | Mini Project A: Naive Bayes on digits (continues Week 7/8/9) | image |
| 5 | Mini Project B: Naive Bayes on the six sentences (continues Week 7/8/9) | NLP |
| 6 | Reflection and capstone link | |

**Rule carried from Week 5:** split first; fit every scaler, PCA and tuned setting on **train only**.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits, load_wine
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, CategoricalNB
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
np.set_printoptions(precision=4, suppress=True)

## Part 1: Probability and Bayes' Theorem
$$P(C\mid x)=\frac{P(x\mid C)\,P(C)}{P(x)}$$
**Posterior** = likelihood × prior / evidence. The evidence $P(x)=\sum_C P(x\mid C)P(C)$ is the same for every class, so to *compare* classes we only need the numerator.

### Predict-then-run
A disease affects 1% of people. A test is positive for 90% of sick people (sensitivity) and also for 5% of healthy people (false-positive rate). You test positive. Chance you are actually sick? Guess before running: **~90% / ~50% / ~15%**

In [21]:
prior, sens, fpr = 0.01, 0.90, 0.05           # P(sick), P(+|sick), P(+|healthy)
evidence = sens * prior + fpr * (1 - prior)    # P(+)
posterior = sens * prior / evidence            # P(sick | +)
print('P(positive)      =', round(evidence, 4))
print('P(sick | positive) =', round(posterior, 4))

# sanity check by simulation: 1,000,000 people
rng = np.random.default_rng(1); n = 1_000_000
sick = rng.random(n) < prior
pos = np.where(sick, rng.random(n) < sens, rng.random(n) < fpr)
print('simulated          =', round(sick[pos].mean(), 4), f'({sick[pos].sum()} sick among {pos.sum()} positives)')

P(positive)      = 0.0585
P(sick | positive) = 0.1538
simulated          = 0.1545 (9035 sick among 58469 positives)


### Student Practice 1
An email filter knows: 40% of mail is spam; the word *free* appears in 50% of spam and 5% of non-spam.
1. Compute $P(\text{spam}\mid \text{free})$ by hand, then in code.
2. If the spam rate were only 5%, what would the posterior be? Why does it drop so much?
3. In the disease example, which term of Bayes' theorem do people usually ignore?

\(P(Spam|Free)=0.8696\), approximately 86.96%.



When the spam rate is 5%, the posterior becomes 34.48%. It drops because spam is less common.



People usually ignore the prior probability \(P(C)\), which represents how common the disease is.

In [23]:
# Q1-Q2: your code here

# Q1: Spam probability
prior_spam = 0.40
p_free_spam = 0.50
p_free_not_spam = 0.05

posterior1 = (prior_spam * p_free_spam) / (
    prior_spam * p_free_spam +
    (1 - prior_spam) * p_free_not_spam
)

print("Q1: P(Spam | Free) =", round(posterior1, 4))

# Q2: Spam rate is only 5%
prior_spam2 = 0.05

posterior2 = (prior_spam2 * p_free_spam) / (
    prior_spam2 * p_free_spam +
    (1 - prior_spam2) * p_free_not_spam
)

print("Q2: P(Spam | Free) =", round(posterior2, 4))

# Explanation:
print("The probability drops because the prior probability of spam is lower.")

Q1: P(Spam | Free) = 0.8696
Q2: P(Spam | Free) = 0.3448
The probability drops because the prior probability of spam is lower.


## Part 2: Naive Bayes by Hand (Play Tennis)
Classic 14-day dataset (Mitchell, Ch. 6). Predict **Play** from four categorical features.

The **naive** assumption: features are independent *given the class*, so
$$P(x_1,\dots,x_d\mid C)=\prod_{j=1}^{d}P(x_j\mid C)\quad\Rightarrow\quad \hat C=\arg\max_C\; P(C)\prod_j P(x_j\mid C)$$

Steps: (1) class priors, (2) conditional probability tables from counts, (3) multiply, (4) normalise, (5) pick the largest.

| Day | Outlook | Temp | Humidity | Wind | Play |
|---|---|---|---|---|---|
| 1 | Sunny | Hot | High | Weak | No |
| 2 | Sunny | Hot | High | Strong | No |
| 3 | Overcast | Hot | High | Weak | Yes |
| 4 | Rain | Mild | High | Weak | Yes |
| 5 | Rain | Cool | Normal | Weak | Yes |
| 6 | Rain | Cool | Normal | Strong | No |
| 7 | Overcast | Cool | Normal | Strong | Yes |
| 8 | Sunny | Mild | High | Weak | No |
| 9 | Sunny | Cool | Normal | Weak | Yes |
| 10 | Rain | Mild | Normal | Weak | Yes |
| 11 | Sunny | Mild | Normal | Strong | Yes |
| 12 | Overcast | Mild | High | Strong | Yes |
| 13 | Overcast | Hot | Normal | Weak | Yes |
| 14 | Rain | Mild | High | Strong | No |

In [4]:
rows = [
 ('Sunny','Hot','High','Weak','No'), ('Sunny','Hot','High','Strong','No'), ('Overcast','Hot','High','Weak','Yes'),
 ('Rain','Mild','High','Weak','Yes'), ('Rain','Cool','Normal','Weak','Yes'), ('Rain','Cool','Normal','Strong','No'),
 ('Overcast','Cool','Normal','Strong','Yes'), ('Sunny','Mild','High','Weak','No'), ('Sunny','Cool','Normal','Weak','Yes'),
 ('Rain','Mild','Normal','Weak','Yes'), ('Sunny','Mild','Normal','Strong','Yes'), ('Overcast','Mild','High','Strong','Yes'),
 ('Overcast','Hot','Normal','Weak','Yes'), ('Rain','Mild','High','Strong','No')]
F = np.array([r[:-1] for r in rows]); y = np.array([r[-1] for r in rows])
features = ['Outlook', 'Temp', 'Humidity', 'Wind']
print('Priors: Yes =', (y == 'Yes').sum(), '/ 14,  No =', (y == 'No').sum(), '/ 14')
for j, name in enumerate(features):
    print(name)
    for v in sorted(set(F[:, j])):
        print(f'   {v:9} Yes {((F[:, j] == v) & (y == "Yes")).sum()}/9    No {((F[:, j] == v) & (y == "No")).sum()}/5')

Priors: Yes = 9 / 14,  No = 5 / 14
Outlook
   Overcast  Yes 4/9    No 0/5
   Rain      Yes 3/9    No 2/5
   Sunny     Yes 2/9    No 3/5
Temp
   Cool      Yes 3/9    No 1/5
   Hot       Yes 2/9    No 2/5
   Mild      Yes 4/9    No 2/5
Humidity
   High      Yes 3/9    No 4/5
   Normal    Yes 6/9    No 1/5
Wind
   Strong    Yes 3/9    No 3/5
   Weak      Yes 6/9    No 2/5


In [5]:
def nb_scores(q, alpha=0):
    """Unnormalised score P(C) * prod_j P(x_j|C) and the normalised posterior. alpha = Laplace smoothing."""
    out = {}
    for c in ['Yes', 'No']:
        Fc = F[y == c]
        p = float((y == c).mean())
        for j, v in enumerate(q):
            k = len(set(F[:, j]))                                  # how many values this feature can take
            p *= float((Fc[:, j] == v).sum() + alpha) / (len(Fc) + alpha * k)
        out[c] = p
    total = sum(out.values())
    return out, {c: out[c] / total for c in out}

### Predict-then-run
Today is **(Sunny, Cool, High, Strong)**. The prior favours *Yes* (9/14). Will Naive Bayes say Yes or No? Guess, then run.

In [6]:
q = ('Sunny', 'Cool', 'High', 'Strong')
scores, post = nb_scores(q)
print('scores   :', {c: round(v, 5) for c, v in scores.items()})
print('posterior:', {c: round(v, 4) for c, v in post.items()})
print('prediction:', max(post, key=post.get))

scores   : {'Yes': 0.00529, 'No': 0.02057}
posterior: {'Yes': 0.2046, 'No': 0.7954}
prediction: No


### Student Practice 2 (do on paper first, as in an exam)
For today = (Sunny, Cool, High, Strong):
1. Write $P(\text{Yes})\,P(\text{Sunny}|\text{Yes})\,P(\text{Cool}|\text{Yes})\,P(\text{High}|\text{Yes})\,P(\text{Strong}|\text{Yes})$ as a product of fractions, then evaluate it. Do the same for *No*.
2. Normalise: $P(\text{Yes}\mid x)$ = ___ %
3. Redo it with Laplace smoothing ($\alpha=1$). Does the prediction change? Does the probability?
4. **Accuracy on test days (Exp 9).** Using $\alpha=1$, classify these five unseen days and report the accuracy:

| Test day | Outlook | Temp | Humidity | Wind | True Play |
|---|---|---|---|---|---|
| T1 | Overcast | Mild | Normal | Weak | Yes |
| T2 | Rain | Hot | High | Strong | No |
| T3 | Sunny | Hot | Normal | Weak | Yes |
| T4 | Rain | Cool | High | Weak | Yes |
| T5 | Sunny | Mild | High | Strong | No |

5. Verify your function against sklearn's `CategoricalNB(alpha=1)` (hint: encode the strings with `OrdinalEncoder`).

In [24]:
tests = [(('Overcast','Mild','Normal','Weak'), 'Yes'), (('Rain','Hot','High','Strong'), 'No'),
         (('Sunny','Hot','Normal','Weak'), 'Yes'), (('Rain','Cool','High','Weak'), 'Yes'),
         (('Sunny','Mild','High','Strong'), 'No')]
# Q1-Q5: your code here

# Q1: Calculate probabilities for the five test days
predictions = []

for q, actual in tests:
    scores, posterior = nb_scores(q, alpha=1)

    prediction = max(posterior, key=posterior.get)
    predictions.append(prediction)

    print("\nTest day:", q)
    print("Actual:", actual)
    print("Scores:", scores)
    print("Posterior:", posterior)
    print("Prediction:", prediction)

# Q2: Accuracy
actual_labels = [actual for q, actual in tests]

accuracy = np.mean(np.array(predictions) == np.array(actual_labels))

print("\nTest Accuracy:", round(accuracy * 100, 2), "%")

# Q3: Verify using sklearn CategoricalNB
encoder = OrdinalEncoder()
F_encoded = encoder.fit_transform(F).astype(int)

test_features = np.array([q for q, actual in tests])
test_encoded = encoder.transform(test_features).astype(int)

model = CategoricalNB(alpha=1)
model.fit(F_encoded, y)

sk_predictions = model.predict(test_encoded)

print("\nSklearn Predictions:", sk_predictions)
print("Our Predictions:", predictions)
print("Both methods match:",
      np.array_equal(sk_predictions, predictions))


Test day: ('Overcast', 'Mild', 'Normal', 'Weak')
Actual: Yes
Scores: {'Yes': 0.04519628099173554, 'No': 0.0020499271137026233}
Posterior: {'Yes': 0.9566118172038727, 'No': 0.04338818279612732}
Prediction: Yes

Test day: ('Rain', 'Hot', 'High', 'Strong')
Actual: No
Scores: {'Yes': 0.007083825265643448, 'No': 0.02049927113702624}
Posterior: {'Yes': 0.25681762345426257, 'No': 0.7431823765457374}
Prediction: No

Test day: ('Sunny', 'Hot', 'Normal', 'Weak')
Actual: Yes
Scores: {'Yes': 0.016270661157024795, 'No': 0.008199708454810493}
Posterior: {'Yes': 0.664912766546663, 'No': 0.3350872334533369}
Prediction: Yes

Test day: ('Rain', 'Cool', 'High', 'Weak')
Actual: Yes
Scores: {'Yes': 0.016528925619834708, 'No': 0.010249635568513119}
Posterior: {'Yes': 0.6172447243740247, 'No': 0.38275527562597533}
Prediction: Yes

Test day: ('Sunny', 'Mild', 'High', 'Strong')
Actual: No
Scores: {'Yes': 0.008854781582054312, 'No': 0.02733236151603498}
Posterior: {'Yes': 0.24469413233458187, 'No': 0.755305867

### Deliberate error: the zero-frequency problem
Today is **(Overcast, Hot, High, Strong)**. *Overcast* never occurs with *No* in the training data. Run the cell, then explain in one line why $\alpha=0$ is dangerous.

In [8]:
q2 = ('Overcast', 'Hot', 'High', 'Strong')
for a in [0, 1]:
    s, p = nb_scores(q2, alpha=a)
    print(f'alpha={a}: scores', {c: round(v, 5) for c, v in s.items()}, ' posterior', {c: round(float(v), 3) for c, v in p.items()})

alpha=0: scores {'Yes': 0.00705, 'No': 0.0}  posterior {'Yes': 1.0, 'No': 0.0}
alpha=1: scores {'Yes': 0.00885, 'No': 0.00683}  posterior {'Yes': 0.564, 'No': 0.436}


## Part 3: Gaussian Naive Bayes (continuous features)
For numeric features, model each feature *within each class* as a bell curve with that class's mean $\mu_{jc}$ and variance $\sigma^2_{jc}$:
$$P(x_j\mid C)=\frac{1}{\sqrt{2\pi\sigma^2_{jc}}}\exp\!\Big(-\frac{(x_j-\mu_{jc})^2}{2\sigma^2_{jc}}\Big)$$
sklearn stores these as `theta_` (means) and `var_` (variances). In practice we add **log-probabilities** instead of multiplying (see the deliberate error below).

In [9]:
Xi, yi = load_iris(return_X_y=True)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, random_state=1, stratify=yi)
gnb = GaussianNB().fit(Xi_tr, yi_tr)
print('class priors:', gnb.class_prior_)
print('class means (rows = setosa, versicolor, virginica):\n', gnb.theta_)
print('class variances:\n', gnb.var_)
print('test accuracy:', round(gnb.score(Xi_te, yi_te), 4))

class priors: [0.3333 0.3333 0.3333]
class means (rows = setosa, versicolor, virginica):
 [[5.0229 3.4343 1.4571 0.2429]
 [5.8714 2.78   4.2743 1.3229]
 [6.6771 2.9943 5.6371 2.0257]]
class variances:
 [[0.1275 0.132  0.0373 0.0076]
 [0.2015 0.0867 0.2042 0.0418]
 [0.3709 0.1125 0.3143 0.0831]]
test accuracy: 0.9778


### Student Practice 3
1. For the first test flower `x = Xi_te[0]`, compute the log-posterior of each class **yourself** from `gnb.class_prior_`, `gnb.theta_`, `gnb.var_` (Gaussian formula, summed over the 4 features). Which class wins? Does it match `gnb.predict`?
2. Repeat for every test row and check that your predictions equal `gnb.predict(Xi_te)`.
3. Print the confusion matrix. Which two species are confused?

The class with the highest manually calculated log-posterior is the predicted class. It matches gnb.predict().



All manually calculated predictions match sklearn predictions.



Versicolor and Virginica are the two species most commonly confused.

In [25]:
# Q1-Q3: your code here

# Q1: Calculate log posterior manually

x = Xi_te[0]

log_scores = []

for c in range(len(gnb.classes_)):

    log_prob = np.log(gnb.class_prior_[c])

    for j in range(Xi.shape[1]):

        mean = gnb.theta_[c, j]
        variance = gnb.var_[c, j]

        log_prob += (
            -0.5 * np.log(2 * np.pi * variance)
            - ((x[j] - mean) ** 2) / (2 * variance)
        )

    log_scores.append(log_prob)

print("Manual log posterior:", log_scores)
print("Manual prediction:", gnb.classes_[np.argmax(log_scores)])
print("Sklearn prediction:", gnb.predict([x])[0])

# Q2: Repeat for every test row

manual_predictions = []

for x in Xi_te:

    scores = []

    for c in range(len(gnb.classes_)):

        log_prob = np.log(gnb.class_prior_[c])

        for j in range(Xi.shape[1]):

            mean = gnb.theta_[c, j]
            variance = gnb.var_[c, j]

            log_prob += (
                -0.5 * np.log(2 * np.pi * variance)
                - ((x[j] - mean) ** 2) / (2 * variance)
            )

        scores.append(log_prob)

    manual_predictions.append(gnb.classes_[np.argmax(scores)])

manual_predictions = np.array(manual_predictions)

sk_predictions = gnb.predict(Xi_te)

print("\nAll predictions match:",
      np.array_equal(manual_predictions, sk_predictions))

# Q3: Confusion matrix

cm = confusion_matrix(yi_te, sk_predictions)

print("\nConfusion Matrix:\n", cm)
print("Class names:", gnb.classes_)

Manual log posterior: [np.float64(-491.1550148130234), np.float64(-17.386548683705605), np.float64(-2.944046930901379)]
Manual prediction: 2
Sklearn prediction: 2

All predictions match: True

Confusion Matrix:
 [[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]
Class names: [0 1 2]


### Deliberate error: multiplying many small probabilities
Predict: what is the product of 200 probabilities, each equal to 0.01? Then run.

In [11]:
p = np.full(200, 0.01)
print('product     :', np.prod(p))
print('sum of logs :', np.sum(np.log(p)))

product     : 0.0
sum of logs : -921.0340371976183


### Predict-then-run
In Week 9, KNN needed scaling on Wine (unscaled PC1 was dominated by *proline*). Does standardising change Gaussian Naive Bayes? Guess: **helps a lot / no change / hurts**. Then run.

In [12]:
Xw, yw = load_wine(return_X_y=True)
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.3, random_state=1, stratify=yw)
sw = StandardScaler().fit(Xw_tr)                       # fit on TRAIN only
Xw_tr_s, Xw_te_s = sw.transform(Xw_tr), sw.transform(Xw_te)
print('model            raw     scaled')
for name, mk in [('GaussianNB', lambda: GaussianNB()), ('KNN (K=3)', lambda: KNeighborsClassifier(3))]:
    a = mk().fit(Xw_tr, yw_tr).score(Xw_te, yw_te)
    b = mk().fit(Xw_tr_s, yw_tr).score(Xw_te_s, yw_te)
    print(f'{name:16} {a:.4f}  {b:.4f}')

model            raw     scaled
GaussianNB       0.9815  0.9815
KNN (K=3)        0.6667  0.9444


### Student Practice 4
1. Explain in one line why Gaussian NB does not care about feature units. (Look at $(x-\mu)^2/\sigma^2$: what happens to it if you change cm to mm?)
2. One split can be lucky. Report GaussianNB accuracy on the full Wine data with 5-fold cross-validation (mean and std). Why is it safe to skip the scaler here?

Gaussian Naive Bayes does not depend on feature units because changing the units scales both the difference from the mean and the standard deviation equally, so the standardised distance remains unchanged.



Results:
- Mean accuracy = 0.9663 (96.63%)
- Standard deviation = 0.0211
Scaling is not necessary because Gaussian NB automatically considers the mean and variance of each feature.

In [26]:
# Q2: your code here

Xw, yw = load_wine(return_X_y=True)

# 5-fold cross-validation
scores = cross_val_score(
    GaussianNB(),
    Xw,
    yw,
    cv=5
)

print("Cross Validation Scores:", scores)
print("Mean Accuracy:", round(scores.mean(), 4))
print("Standard Deviation:", round(scores.std(), 4))

Cross Validation Scores: [0.9444 0.9722 0.9722 0.9429 1.    ]
Mean Accuracy: 0.9663
Standard Deviation: 0.0211


# Mini Project A: Naive Bayes on Digits (Image Track — continuing from Weeks 7, 8 and 9)
Same split as before (`test_size=0.3, random_state=1, stratify`). Baselines you already have: **KNN (scaled, K=3) = 0.9796** and **Tree (entropy, full depth, unscaled) = 0.8815**. Question: how does Naive Bayes compare, and does Week 9's PCA help it?

In [14]:
digits = load_digits()
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=1, stratify=digits.target)
print('pixels with zero variance in train:', int((Xd_train.std(0) == 0).sum()), 'of 64')
print('GaussianNB test accuracy:', round(GaussianNB().fit(Xd_train, yd_train).score(Xd_test, yd_test), 4))

pixels with zero variance in train: 3 of 64
GaussianNB test accuracy: 0.8315


### Predict-then-run
Before running the cell above, guess: will GaussianNB score **above both** baselines, **between** them, or **below both**? The 64 pixels are clearly *not* independent given the digit. Is that a problem for NB?

### Student Practice 5
1. Fit `MultinomialNB()` (treat pixel intensity as a count) and `BernoulliNB(binarize=8)` (ink / no ink) on the raw pixels. Report test accuracy for both.
2. GaussianNB has one knob, `var_smoothing`. For values `[1e-9, 1e-3, 1e-2, 1e-1, 1]`, use 5-fold cross-validation **on the training set only**, pick the best, then report its test accuracy **once**. (Why not pick the value with the best test accuracy?)
3. Using that best model, which two digits are confused most often?

In [27]:
# Q1-Q3: your code here

# Q1: MultinomialNB
mnb = MultinomialNB()
mnb.fit(Xd_train, yd_train)

mnb_accuracy = mnb.score(Xd_test, yd_test)

print("MultinomialNB Accuracy:",
      round(mnb_accuracy, 4))

# BernoulliNB
bnb = BernoulliNB(binarize=8)
bnb.fit(Xd_train, yd_train)

bnb_accuracy = bnb.score(Xd_test, yd_test)

print("BernoulliNB Accuracy:",
      round(bnb_accuracy, 4))


# Q2: Find best var_smoothing using 5-fold CV

values = [1e-9, 1e-3, 1e-2, 1e-1, 1]

best_score = 0
best_value = None

for value in values:

    model = GaussianNB(var_smoothing=value)

    scores = cross_val_score(
        model,
        Xd_train,
        yd_train,
        cv=5
    )

    mean_score = scores.mean()

    print("Var smoothing:", value,
          "CV Accuracy:", round(mean_score, 4))

    if mean_score > best_score:
        best_score = mean_score
        best_value = value

print("\nBest Var Smoothing:", best_value)


# Evaluate best model on test data once

best_model = GaussianNB(var_smoothing=best_value)
best_model.fit(Xd_train, yd_train)

test_accuracy = best_model.score(Xd_test, yd_test)

print("Best Model Test Accuracy:",
      round(test_accuracy, 4))


# Q3: Confusion matrix

predictions = best_model.predict(Xd_test)

cm = confusion_matrix(yd_test, predictions)

print("\nConfusion Matrix:\n", cm)

# Find most confused pair
cm_no_diagonal = cm.copy()
np.fill_diagonal(cm_no_diagonal, 0)

i, j = np.unravel_index(
    np.argmax(cm_no_diagonal),
    cm_no_diagonal.shape
)

print("Most confused digits:", i, "and", j)
print("Confusion count:", cm_no_diagonal[i, j])

MultinomialNB Accuracy: 0.8963
BernoulliNB Accuracy: 0.8963
Var smoothing: 1e-09 CV Accuracy: 0.8187
Var smoothing: 0.001 CV Accuracy: 0.903
Var smoothing: 0.01 CV Accuracy: 0.9173
Var smoothing: 0.1 CV Accuracy: 0.9141
Var smoothing: 1 CV Accuracy: 0.9046

Best Var Smoothing: 0.01
Best Model Test Accuracy: 0.9296

Confusion Matrix:
 [[52  0  0  0  2  0  0  0  0  0]
 [ 0 44  4  0  0  0  0  0  6  1]
 [ 0  0 51  1  0  0  0  0  1  0]
 [ 0  0  0 50  0  0  0  2  2  1]
 [ 0  0  0  0 50  0  0  3  1  0]
 [ 0  0  0  1  0 53  1  0  0  0]
 [ 0  0  0  0  2  0 52  0  0  0]
 [ 0  0  0  0  1  0  0 52  1  0]
 [ 0  4  0  0  0  1  0  1 46  0]
 [ 0  0  1  0  0  0  0  1  0 52]]
Most confused digits: 1 and 8
Confusion count: 6


### Deliberate error: negative inputs
Since Week 9 we standardise pixels before PCA. Run the cell: what breaks, and why?

In [16]:
sc = StandardScaler().fit(Xd_train)                    # fit on TRAIN only
Xtr_s, Xte_s = sc.transform(Xd_train), sc.transform(Xd_test)
try:
    MultinomialNB().fit(Xtr_s, yd_train)
except ValueError as e:
    print('ValueError:', e)
print('GaussianNB on scaled pixels:', round(GaussianNB().fit(Xtr_s, yd_train).score(Xte_s, yd_test), 4), ' (raw pixels gave 0.8315)')

ValueError: Negative values in data passed to MultinomialNB (input X).
GaussianNB on scaled pixels: 0.7759  (raw pixels gave 0.8315)


### Link to Week 9: does decorrelating the pixels help?
Naive Bayes assumes features are independent given the class. PCA scores are *uncorrelated by construction*. Question: does PCA before NB help?

### Student Practice 6
1. Compute the mean absolute off-diagonal correlation (a) of the non-constant raw pixels and (b) of the first 30 PCA scores of `Xtr_s`.
2. For k in `[10, 20, 30, 40, 64]` fit `PCA(k)` on `Xtr_s` (train only), then `GaussianNB` on the scores, and report test accuracy.
3. `PCA(64)` is only a rotation of the scaled pixels: no information is lost. So why does its accuracy differ from GaussianNB on scaled pixels (above)?

In [28]:
# Q1-Q3: your code here

# Q1: Mean absolute off-diagonal correlation

# Remove constant pixels
non_constant = Xtr_s[:, Xtr_s.std(axis=0) > 0]

def mean_off_diagonal_correlation(data):

    corr = np.corrcoef(data, rowvar=False)

    mask = ~np.eye(corr.shape[0], dtype=bool)

    return np.mean(np.abs(corr[mask]))

raw_corr = mean_off_diagonal_correlation(non_constant)

print("Raw Pixel Correlation:", round(raw_corr, 4))


# PCA with 30 components
pca30 = PCA(n_components=30)

Xtr_pca30 = pca30.fit_transform(Xtr_s)

pca_corr = mean_off_diagonal_correlation(Xtr_pca30)

print("PCA Correlation:", round(pca_corr, 4))


# Q2: Test different PCA components

for k in [10, 20, 30, 40, 64]:

    pca = PCA(n_components=k)

    X_train_pca = pca.fit_transform(Xtr_s)
    X_test_pca = pca.transform(Xte_s)

    model = GaussianNB()

    model.fit(X_train_pca, yd_train)

    accuracy = model.score(X_test_pca, yd_test)

    print("PCA Components:", k,
          "Accuracy:", round(accuracy, 4))

Raw Pixel Correlation: 0.1257
PCA Correlation: 0.0
PCA Components: 10 Accuracy: 0.8296
PCA Components: 20 Accuracy: 0.9148
PCA Components: 30 Accuracy: 0.9259
PCA Components: 40 Accuracy: 0.9222
PCA Components: 64 Accuracy: 0.9185


# Mini Project B: Naive Bayes on the Six Sentences (NLP Track — continuing from Weeks 7, 8 and 9)
Same sentences and bag-of-words vectors. Naive Bayes is the classic text classifier (spam filters!). In Week 9, PCA failed to find the topics because *the* and *and* dominated the variance. Question: does NB do better?

In [18]:
sentences = [
    'the cricket team won the match',
    'the batsman scored a century today',
    'chop the onions and boil the pasta',
    'add salt and simmer the curry',
    'the new phone has a faster processor',
    'the laptop battery lasts all day',
]
labels = ['sports', 'sports', 'cooking', 'cooking', 'tech', 'tech']
vocab = sorted(set(w for s in sentences for w in s.lower().split()))
def vec(s): return [s.lower().split().count(w) for w in vocab]       # words outside the vocabulary are ignored
X_text = np.array([vec(s) for s in sentences])
print('matrix shape (docs x vocab):', X_text.shape)
nb_text = MultinomialNB(alpha=1).fit(X_text, labels)
print('classes:', nb_text.classes_, ' priors:', np.exp(nb_text.class_log_prior_))

matrix shape (docs x vocab): (6, 29)
classes: ['cooking' 'sports' 'tech']  priors: [0.3333 0.3333 0.3333]


### Predict-then-run
Multinomial NB with smoothing uses $P(w\mid C)=\dfrac{\text{count}(w,C)+\alpha}{\text{total words in }C+\alpha\,|V|}$ with $|V|=29$.
(a) How many word tokens are in the *sports* class (two sentences)? (b) Which class will NB give to **'the new curry'**? Is it confident?

### Student Practice 7
1. Predict the five sentences below with `nb_text` (`alpha=1`) and print the class probabilities. Accuracy?
   `'the team scored a century'` (sports), `'boil the pasta with salt'` (cooking), `'the phone battery lasts a day'` (tech), `'the match today'` (sports), `'the new curry'` (cooking)
2. By hand: for `'the team scored a century'` compute the three class scores from the count formula above and normalise. Compare with `predict_proba`.
3. Print the 5 words with the highest $P(w\mid C)$ in each class. Is *the* near the top everywhere? Why does that **not** hurt Naive Bayes, when it did hurt PCA?
4. Why is the probability for **'the new curry'** so spread out?

In [29]:
# Q1-Q4: your code here

# Q1: Predict five sentences

test_sentences = [
    'the team scored a century',
    'boil the pasta with salt',
    'the phone battery lasts a day',
    'the match today',
    'the new curry'
]

actual_labels = [
    'sports',
    'cooking',
    'tech',
    'sports',
    'cooking'
]

X_test_text = np.array([vec(s) for s in test_sentences])

predictions = nb_text.predict(X_test_text)

probabilities = nb_text.predict_proba(X_test_text)

for sentence, prediction, probability in zip(
    test_sentences, predictions, probabilities
):

    print("\nSentence:", sentence)
    print("Prediction:", prediction)
    print("Probabilities:",
          dict(zip(nb_text.classes_, probability.round(4))))


accuracy = np.mean(predictions == actual_labels)

print("\nAccuracy:", round(accuracy * 100, 2), "%")


# Q2: Calculate class scores manually

sentence = 'the team scored a century'

word_counts = np.array(vec(sentence))

manual_scores = {}

for i, c in enumerate(nb_text.classes_):

    log_score = nb_text.class_log_prior_[i]

    log_score += np.sum(
        word_counts * nb_text.feature_log_prob_[i]
    )

    manual_scores[c] = np.exp(log_score)

total = sum(manual_scores.values())

manual_probabilities = {
    c: score / total
    for c, score in manual_scores.items()
}

print("\nManual Scores:", manual_scores)
print("Manual Probabilities:", manual_probabilities)

print("Sklearn Probabilities:",
      nb_text.predict_proba([vec(sentence)])[0])


# Q3: Top 5 words in each class

for i, c in enumerate(nb_text.classes_):

    top_indices = np.argsort(
        nb_text.feature_log_prob_[i]
    )[::-1][:5]

    print("\nClass:", c)

    for index in top_indices:

        print(
            vocab[index],
            round(
                np.exp(nb_text.feature_log_prob_[i, index]),
                4
            )
        )


# Q4: Probability for 'the new curry'

print("\nProbability for 'the new curry':")

print(
    dict(
        zip(
            nb_text.classes_,
            nb_text.predict_proba([vec('the new curry')])[0].round(4)
        )
    )
)


Sentence: the team scored a century
Prediction: sports
Probabilities: {np.str_('cooking'): np.float64(0.0487), np.str_('sports'): np.float64(0.8783), np.str_('tech'): np.float64(0.073)}

Sentence: boil the pasta with salt
Prediction: cooking
Probabilities: {np.str_('cooking'): np.float64(0.8121), np.str_('sports'): np.float64(0.1118), np.str_('tech'): np.float64(0.0761)}

Sentence: the phone battery lasts a day
Prediction: tech
Probabilities: {np.str_('cooking'): np.float64(0.0366), np.str_('sports'): np.float64(0.0846), np.str_('tech'): np.float64(0.8788)}

Sentence: the match today
Prediction: sports
Probabilities: {np.str_('cooking'): np.float64(0.1653), np.str_('sports'): np.float64(0.7107), np.str_('tech'): np.float64(0.124)}

Sentence: the new curry
Prediction: cooking
Probabilities: {np.str_('cooking'): np.float64(0.4372), np.str_('sports'): np.float64(0.235), np.str_('tech'): np.float64(0.3279)}

Accuracy: 100.0 %

Manual Scores: {np.str_('cooking'): np.float64(1.0202163625891

### Deliberate error: almost no smoothing
`'the cricket phone'` mixes a sports word with a tech word. Run the cell and compare how *confident* each model is.

In [20]:
for a in [1e-10, 1.0]:
    m = MultinomialNB(alpha=a).fit(X_text, labels)
    print(f'alpha={a:<6}', dict(zip(m.classes_.tolist(), m.predict_proba([vec('the cricket phone')])[0].round(3).tolist())))

alpha=1e-10  {'cooking': 0.0, 'sports': 0.656, 'tech': 0.344}
alpha=1.0    {'cooking': 0.215, 'sports': 0.462, 'tech': 0.323}


## Part 6: Reflection

**1. What is the naive assumption, and what does it make easy?**

Naive Bayes assumes that all features are independent of each other given the class. This makes calculations easier because we can multiply individual feature probabilities instead of calculating joint probabilities.

**2. What is the zero-frequency problem?**

The zero-frequency problem occurs when a feature value never appears in a particular class. In the Play Tennis example, Overcast never appears with No. Therefore, its probability becomes zero. Laplace smoothing adds 1 to the counts to avoid zero probabilities.

**3. Why do real implementations add log-probabilities instead of multiplying probabilities?**

Probabilities can be very small, and multiplying many small values can cause numerical underflow. Using logarithms converts multiplication into addition and makes calculations more stable.

**4. Wine: scaling changed KNN but not Gaussian NB. What does that tell you?**

KNN depends on distances between data points, so feature scaling is important. Gaussian NB uses the mean and variance of each feature, so it automatically handles different feature units.

**5. Digits: PCA helped NB but hurt the tree. Explain using the independence assumption.**

Naive Bayes assumes that features are independent. PCA creates uncorrelated features, which can help Naive Bayes. Decision trees can work directly with original features, and PCA may remove useful feature information for them.

**6. When would you still choose Naive Bayes?**

I would choose Naive Bayes for text classification, spam detection and sentiment analysis. It is fast, requires less training data and works well with high-dimensional datasets.

**7. Capstone link: Will Naive Bayes belong in your pipeline?**

Yes. I would include Naive Bayes as a baseline model in my machine learning pipeline. I would compare its accuracy with other classifiers to understand which model performs better on my dataset.

*Submit this notebook to GitHub under `labs/week10_naive_bayes/`.*